# 🌿 CropGuard AI — YOLOv8 Multi-Crop Disease Detection
**38 Classes | 14 Crop Species | 54,000+ Images | Target: 96%+ Accuracy**

### Supported Crops & Diseases
| Crop | Diseases |
|------|----------|
| 🍎 Apple | Scab, Black Rot, Cedar Rust, Healthy |
| 🍒 Cherry | Powdery Mildew, Healthy |
| 🌽 Corn | Gray Leaf Spot, Common Rust, Northern Leaf Blight, Healthy |
| 🍇 Grape | Black Rot, Esca, Leaf Blight, Healthy |
| 🍊 Orange | Citrus Greening (HLB) |
| 🍑 Peach | Bacterial Spot, Healthy |
| 🌶️ Pepper | Bacterial Spot, Healthy |
| 🥔 Potato | Early Blight, Late Blight, Healthy |
| 🧆 Raspberry | Healthy |
| 🫅 Soybean | Healthy |
| 🍏 Squash | Powdery Mildew |
| 🍓 Strawberry | Leaf Scorch, Healthy |
| 🍅 Tomato | Bacterial Spot, Early/Late Blight, Leaf Mold, Septoria, Spider Mites, Target Spot, YLCV, Mosaic Virus, Healthy |
| 🪷 Blueberry | Healthy |

---
⚡ **Runtime → Change runtime type → T4 GPU** before running!

In [ ]:
# ═══ CELL 1: Install Dependencies ═══════════════════════════════
!pip install -q ultralytics>=8.2.0 albumentations>=1.3.0 datasets Pillow seaborn scikit-learn
!pip install -q onnx tensorflowjs

import torch, os, json
print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ═══ CELL 2: Download PlantVillage (all 14 crops, 38 classes) ═══
import os, shutil, random, json
from pathlib import Path
from datasets import load_dataset

DATA_DIR = Path('data')
RAW_DIR = DATA_DIR / 'raw' / 'plantvillage'
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Skip if already downloaded
existing = list(RAW_DIR.rglob('*.jpg'))
if len(existing) > 1000:
    print(f'\u2705 Already downloaded ({len(existing)} images). Skipping.')
else:
    print('\ud83d\udce6 Downloading PlantVillage from Hugging Face...')
    ds = load_dataset('mohanty/PlantVillage', split='train')
    label_names = ds.features['label'].names
    print(f'Found {len(label_names)} classes, {len(ds)} images')

    for idx, sample in enumerate(ds):
        label_name = label_names[sample['label']]
        clean = label_name.replace(' ', '_')
        d = RAW_DIR / clean
        d.mkdir(parents=True, exist_ok=True)
        sample['image'].save(d / f'{idx:06d}.jpg')
        if (idx + 1) % 10000 == 0:
            print(f'  Saved {idx+1}/{len(ds)}...')

    total = len(list(RAW_DIR.rglob('*.jpg')))
    print(f'\n\u2705 Downloaded {total} images in {len(label_names)} classes')

# Show all classes
classes = sorted([d.name for d in RAW_DIR.iterdir() if d.is_dir()])
print(f'\n\ud83c\udf3f {len(classes)} disease classes:')
for i, c in enumerate(classes):
    count = len(list((RAW_DIR / c).glob('*.jpg')))
    print(f'  {i:2d}. {c} ({count} imgs)')

In [ ]:
# ═══ CELL 3: Split into Train / Val / Test (75/15/10) ═══════
DATASET_DIR = DATA_DIR / 'dataset'
random.seed(42)
VAL_RATIO, TEST_RATIO = 0.15, 0.10

# Skip if already split
if (DATASET_DIR / 'train').exists() and len(list((DATASET_DIR / 'train').rglob('*.jpg'))) > 1000:
    n = len(list((DATASET_DIR / 'train').rglob('*.jpg')))
    print(f'\u2705 Already split ({n} training images). Skipping.')
else:
    class_dirs = sorted([d for d in RAW_DIR.iterdir() if d.is_dir()])
    stats = {'train': 0, 'val': 0, 'test': 0}

    for cd in class_dirs:
        imgs = list(cd.glob('*.jpg'))
        if not imgs: continue
        random.shuffle(imgs)
        n = len(imgs)
        nv = max(1, int(n * VAL_RATIO))
        nt = max(1, int(n * TEST_RATIO))
        splits = {'train': imgs[:n-nv-nt], 'val': imgs[n-nv-nt:n-nt], 'test': imgs[n-nt:]}
        for sp, im_list in splits.items():
            dest = DATASET_DIR / sp / cd.name
            dest.mkdir(parents=True, exist_ok=True)
            for img in im_list:
                shutil.copy2(img, dest / img.name)
            stats[sp] += len(im_list)

    print('Dataset split:')
    for sp, ct in stats.items():
        print(f'  {sp}: {ct}')
    print(f'  Total: {sum(stats.values())}')

In [ ]:
# ═══ CELL 4: Train YOLOv8m-cls (all 38 classes, all crops) ═══
# ~30-60 min on Colab T4. Targets 96%+ accuracy.
from ultralytics import YOLO

model = YOLO('yolov8m-cls.pt')  # 12M params, ImageNet pretrained

results = model.train(
    data=str(DATASET_DIR),
    epochs=80,
    imgsz=224,
    batch=64,
    # Optimizer
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    cos_lr=True,
    warmup_epochs=5,
    # Early stopping
    patience=15,
    # Augmentation
    augment=True,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,
    degrees=15.0,
    translate=0.1,
    scale=0.3,
    fliplr=0.5,
    flipud=0.2,
    erasing=0.1,
    # Output
    project='runs',
    name='cropguard_v1',
    exist_ok=True,
    plots=True,
    verbose=True,
    seed=42,
)
print('\n\u2705 Training complete!')

In [ ]:
# ═══ CELL 5: Evaluate on Val + Test sets ═════════════════
from ultralytics import YOLO

best = YOLO('runs/cropguard_v1/weights/best.pt')

print('\ud83d\udcca Validation:')
val_r = best.val(data=str(DATASET_DIR), split='val', plots=True)
print(f'  Top-1: {val_r.top1*100:.2f}%  |  Top-5: {val_r.top5*100:.2f}%')

print('\n\ud83d\udcca Test:')
test_r = best.val(data=str(DATASET_DIR), split='test', plots=True)
print(f'  Top-1: {test_r.top1*100:.2f}%  |  Top-5: {test_r.top5*100:.2f}%')

if test_r.top1 >= 0.96:
    print(f'\n\ud83c\udfaf TARGET MET! {test_r.top1*100:.2f}% >= 96%')
else:
    print(f'\n\u26a0\ufe0f {test_r.top1*100:.2f}% < 96%. Run Cell 5b for a larger model.')

In [ ]:
# ═══ CELL 5b: Per-Class Accuracy & Confusion Matrix ═══════
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from PIL import Image
from collections import defaultdict

best = YOLO('runs/cropguard_v1/weights/best.pt')
test_dir = DATASET_DIR / 'test'
class_dirs = sorted([d for d in test_dir.iterdir() if d.is_dir()])

# Per-class accuracy
per_class = {}
crop_acc = defaultdict(lambda: {'correct': 0, 'total': 0})

for cd in class_dirs:
    imgs = list(cd.glob('*.jpg'))[:50]  # sample for speed
    correct = 0
    for img in imgs:
        r = best.predict(str(img), verbose=False)[0]
        if r.names[r.probs.top1] == cd.name:
            correct += 1
    acc = correct / len(imgs) if imgs else 0
    per_class[cd.name] = {'acc': acc, 'n': len(imgs), 'correct': correct}
    crop = cd.name.split('___')[0]
    crop_acc[crop]['correct'] += correct
    crop_acc[crop]['total'] += len(imgs)

print('\ud83c\udf3f Per-Crop Accuracy:')
print(f'{"Crop":<30} {"Accuracy":>10} {"Samples":>8}')
print('-' * 50)
for crop, data in sorted(crop_acc.items()):
    a = data['correct'] / data['total'] * 100
    print(f'{crop:<30} {a:>9.1f}% {data["total"]:>7}')

print(f'\n\ud83c\udf3f Per-Class Accuracy (worst 5):')
worst = sorted(per_class.items(), key=lambda x: x[1]['acc'])[:5]
for name, d in worst:
    print(f'  {name}: {d["acc"]*100:.1f}% ({d["correct"]}/{d["n"]})')

In [ ]:
# ═══ CELL 5c: (OPTIONAL) Retrain with larger model if <96% ═
# Uncomment ONLY if Cell 5 showed < 96%

# from ultralytics import YOLO
# model_l = YOLO('yolov8l-cls.pt')  # 37M params
# model_l.train(
#     data=str(DATASET_DIR), epochs=100, imgsz=224, batch=32,
#     optimizer='AdamW', lr0=0.0005, lrf=0.01, cos_lr=True,
#     warmup_epochs=5, patience=20, augment=True,
#     project='runs', name='cropguard_v2_large',
#     exist_ok=True, plots=True, seed=42,
# )

In [ ]:
# ═══ CELL 6: Visual Predictions Grid ═════════════════════
import matplotlib.pyplot as plt
from PIL import Image
import random

best = YOLO('runs/cropguard_v1/weights/best.pt')
test_images = list((DATASET_DIR / 'test').rglob('*.jpg'))
samples = random.sample(test_images, min(12, len(test_images)))

fig, axes = plt.subplots(3, 4, figsize=(18, 14))
for ax, ip in zip(axes.flat, samples):
    r = best.predict(str(ip), verbose=False)[0]
    pred = r.names[r.probs.top1]
    conf = r.probs.top1conf.item()
    true = ip.parent.name
    img = Image.open(ip)
    ax.imshow(img)
    color = 'green' if pred == true else 'red'
    ax.set_title(f'P: {pred.split("___")[1][:20]}\nT: {true.split("___")[1][:20]}\n{conf:.0%}',
                 fontsize=8, color=color)
    ax.axis('off')
plt.tight_layout()
plt.savefig('runs/cropguard_v1/sample_predictions.png', dpi=150)
plt.show()

In [ ]:
# ═══ CELL 7: Export Model (ONNX + TorchScript) ═════════════
from ultralytics import YOLO

best = YOLO('runs/cropguard_v1/weights/best.pt')

print('\ud83d\udce6 Exporting ONNX...')
best.export(format='onnx', imgsz=224, simplify=True)
print('\u2705 ONNX done')

print('\ud83d\udce6 Exporting TorchScript...')
best.export(format='torchscript', imgsz=224)
print('\u2705 TorchScript done')

!ls -la runs/cropguard_v1/weights/

In [ ]:
# ═══ CELL 8: Package & Download ══════════════════════════
import shutil, json
from pathlib import Path
from ultralytics import YOLO

best = YOLO('runs/cropguard_v1/weights/best.pt')
export_dir = Path('cropguard_model')
export_dir.mkdir(exist_ok=True)

# Copy weights
shutil.copy2('runs/cropguard_v1/weights/best.pt', export_dir / 'best.pt')
try:
    shutil.copy2('runs/cropguard_v1/weights/best.onnx', export_dir / 'best.onnx')
except: pass

# Save class names (this is what the Next.js app reads)
with open(export_dir / 'class_names.json', 'w') as f:
    json.dump(best.names, f, indent=2)

# Copy plots
for p in Path('runs/cropguard_v1').glob('*.png'):
    shutil.copy2(p, export_dir / p.name)

# Zip it
shutil.make_archive('cropguard_model', 'zip', '.', 'cropguard_model')

# Auto-download in Colab
try:
    from google.colab import files
    files.download('cropguard_model.zip')
    print('\u2705 Download started!')
except:
    print('\u2705 Ready: cropguard_model.zip (download from file browser)')

print('\n\ud83d\udce6 Package contents:')
for f in sorted(export_dir.iterdir()):
    print(f'  {f.name} ({f.stat().st_size / 1024:.0f} KB)')

---
## 📝 Integration with CropGuard Next.js App

After training:
1. Download `cropguard_model.zip`
2. Place `best.pt` in your project root
3. `class_names.json` maps index → class name (matches `constants.js`)
4. Set `MODEL_READY = true` in `src/lib/model.js`

### All 38 Classes Supported
```
Apple (4) | Blueberry (1) | Cherry (2) | Corn (4)
Grape (4) | Orange (1) | Peach (2) | Pepper (2)
Potato (3) | Raspberry (1) | Soybean (1) | Squash (1)
Strawberry (2) | Tomato (10)
```

### If Accuracy < 96%
- Run Cell 5c (yolov8l-cls — 37M params)
- Increase epochs to 120
- Reduce batch size to 32 if OOM